# Lab 5: Interview Practice Agent

You're going to build an agent that **role-plays as a job interviewer**.

The agent will:
1. **Read** about you from a file
2. **Read** your resume
3. **Read** a job posting
4. **Conduct** a realistic job interview with you

This builds on Lab 4 — same tools pattern, but now your agent has a **persona** and engages in **multi-turn conversation**.

---

## What's New in This Lab?

In Lab 4, your agent was a **helper** — it did a task and returned a result.

In Lab 5, your agent is a **character** — it plays a role and has a conversation with you.

| Lab 4 | Lab 5 |
|-------|-------|
| One-shot task | Multi-turn conversation |
| Helper persona | Interviewer persona |
| "Find me jobs" | "Interview me for this job" |
| Returns a result | Has a back-and-forth dialogue |

---

## Setup

Run these cells to get started.

In [1]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://pypi.fury.io/ericmichael/
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

True

In [3]:
from agents import Agent
from omniagents import Runner, function_tool
from markitdown import MarkItDown

---

## Part 1: Create Your Tools

You'll reuse two tools from Lab 4 and add one new one.

**Your job:** Complete the three tools below.

In [4]:
@function_tool
def read_job_posting() -> str:
    """Read the job posting — the role, requirements, and responsibilities."""
    # YOUR TURN: Read and return the contents of examples/lab5/job_posting.md
    
    with open("examples/lab5/job_posting.md") as f:
        return f.read()

@function_tool
def read_about_me() -> str:
    """Read information about the user — who they are, what they're looking for, and their preferences."""
    # YOUR TURN: Read and return the contents of examples/lab4/about_me.md
    with open("examples/lab4/about_me.md") as f:
        return f.read()


@function_tool
def read_resume() -> str:
    """Read the user's resume from a docx document — their education, experience, and skills."""
    # YOUR TURN: Convert examples/lab4/resume.docx to markdown and return its contents
    # Use MarkItDown to convert the DOCX document
    md = MarkItDown()
    result = md.convert("examples/lab4/resume.docx")
    return result.text_content

**Hints:**

To read a markdown file:
```python
with open("path/to/file.md") as f:
    return f.read()
```

To convert a Word document to markdown:
```python
md = MarkItDown()
result = md.convert("path/to/file.docx")
return result.text_content
```

---

## Part 2: The Prompt Template

In Lab 4, you wrote instructions as a simple list of steps. For persona-based agents, we use a more structured template:

```
You are a [ROLE]...

## Identity / Personality
Who is this agent? What's their background? How do they behave?

## Goal
What is this agent trying to accomplish?

## Starting Context
What should the agent do FIRST before engaging? (hint: read the files!)

## Guidance
How should the agent handle the conversation? What patterns should it follow?

## Response Style
What tone? How long should responses be? Any formatting rules?
```

This template helps you think through all the dimensions of a persona. **We've pre-filled most of it for you** — just review it and tweak if you want.

In [5]:
INSTRUCTIONS = """
You are a hiring manager conducting a job interview for a tech company.

## Identity / Personality
You are Sarah Chen, Senior Engineering Manager at Nexus AI. You've been with the company 
for 4 years and have hired over 20 engineers. You're known for being warm but thorough — 
you put candidates at ease while still asking substantive questions. You genuinely enjoy 
meeting new people and learning about their experiences.

## Goal
Conduct a realistic 15-20 minute job interview to assess if the candidate is a good fit 
for the role. You want to understand their technical skills, problem-solving ability, 
and whether they'd thrive on your team.

## Starting Context
Before asking ANY questions, you MUST:
1. Use read_job_posting to understand the role you're hiring for
2. Use read_resume to review the candidate's background
3. Use read_about_me to understand what they're looking for

Only after reading all three should you begin the interview.

## Guidance
- Start with a warm introduction — say your name, your role, and give a brief overview of the interview
- Ask 4-5 questions total, mixing:
  - Technical questions related to the job requirements
  - Behavioral questions ("Tell me about a time when...")
  - Questions that reference SPECIFIC items from their resume
- Listen actively — ask follow-up questions based on their answers
- If an answer is vague, probe deeper ("Can you give me a specific example?")
- End by asking if they have questions for you, then explain next steps

## Response Style
- Conversational and warm, but professional
- Keep responses concise — this is a dialogue, not a monologue
- Use the candidate's name occasionally
- React naturally to their answers ("That's interesting!", "I see", etc.)
"""

---

## Part 3: Create the Agent

Create the agent with:
- A name (maybe something like "Hiring Manager" or "Technical Interviewer")
- Your instructions
- All three tools
- A model (use `gpt-5.2`)

In [6]:
interviewer = Agent(
    name="Interviewer",  # YOUR TURN: Give it a name
    instructions=INSTRUCTIONS,
    tools=[read_job_posting, read_about_me, read_resume],  # YOUR TURN: What tools does it need? (hint: 3 tools)
    model="gpt-5.2",
)

---

## Part 4: Practice Your Interview!

Run your agent and have a conversation. This is **multi-turn** — you'll go back and forth with the interviewer.

Try to:
- Answer questions as yourself
- See how the interviewer responds
- Notice if it stays in character

In [8]:
runner = Runner.from_agent(interviewer)
runner.run_notebook(input="Hi! I'm ready for my interview.")

---

## Part 5: How Would You Grade This?

In Lab 4, we had an auto-grader that checked if specific tools were called. Easy.

But how would you auto-grade an interview agent? Think about it:

| What we'd want to check | Why it's hard to automate |
|-------------------------|---------------------------|
| Did it stay in character? | Requires understanding persona consistency |
| Were the questions relevant? | Requires judging question quality |
| Did it respond appropriately? | Requires understanding context |
| Was the tone right? | Subjective and nuanced |
| Did it feel like a real interview? | Holistic judgment |

**This is a real challenge in AI.** Evaluating agents that have conversations, personas, or subjective quality is genuinely hard. There's no simple `assert` statement for "good interview."

### Approaches People Use

1. **Human evaluation** — have people rate the conversations (expensive, slow)
2. **LLM-as-judge** — use another AI to evaluate (can work, but has biases)
3. **Proxy metrics** — count questions asked, measure response length, etc. (easy but incomplete)
4. **User satisfaction** — ask the user if it was helpful (practical but noisy)

For now, **you are the evaluator**. Did your agent feel like a real interviewer?

---

## Part 6: Iterate and Improve

The prompt template is pre-filled, but you can tweak any section to change the agent's behavior.

### Quick Experiments to Try

**Change the personality:**
- Make Sarah more formal: "You maintain professional distance and focus purely on qualifications"
- Make her more casual: "You're known for your relaxed interview style — you often joke around"

**Change the question style:**
- Add: "Focus heavily on technical depth — ask candidates to explain their code decisions"
- Or: "Focus on culture fit — ask about teamwork, conflict resolution, and communication"

**Change the response style:**
- "Be very brief — one short paragraph max per response"
- "Think out loud — share what you're evaluating as you ask questions"

### Common Issues

**"The agent doesn't read the files first"**
- Check that your tools are implemented correctly (they should return file contents, not `None`)
- Check that you included all 3 tools in the `tools=[]` list

**"The agent asks generic questions"**
- The Starting Context section tells it to read the files — make sure your tools work

**"The agent breaks character"**
- Add to Response Style: "Never break character. You ARE Sarah Chen, not an AI assistant."

---

## Part 7: Make It Yours

Once your agent works well:

1. Update `examples/lab4/about_me.md` with your real info (if you haven't already)
2. Replace `examples/lab4/resume.docx` with your real resume
3. Find a real job posting you're interested in and paste it into `examples/lab5/job_posting.md`
4. Practice interviewing for a job you actually want!

---

## Challenge (Optional)

Try modifying the **Identity / Personality** section to create different interviewer types:

**The Tough Technical Interviewer:**
```
You are Marcus Webb, Principal Engineer. You've seen a lot of candidates who can't 
back up their resumes. You ask precise technical questions and push back when answers 
are vague. You're not mean, but you don't let anything slide.
```

**The Friendly HR Screener:**
```
You are Jamie Park, People Operations Lead. Your job is to assess culture fit and 
communication skills, not technical depth. You're warm, encouraging, and focused on 
understanding the person behind the resume.
```

**The Startup Founder:**
```
You are Alex Rivera, CEO of a 10-person startup. You're looking for someone who can 
wear many hats. You care less about credentials and more about hustle, learning speed, 
and whether they'd thrive in ambiguity.
```

How much does the personality section change the conversation?

---

## What You Just Learned

You leveled up from Lab 4:

| Lab 4 | Lab 5 |
|-------|-------|
| Agent as helper | Agent as character |
| One-shot task | Multi-turn conversation |
| Easy to auto-grade | Hard to auto-grade |
| Instructions describe workflow | Instructions describe persona + behavior |

Key insights:
- **Persona matters** — the same tools can feel completely different with different instructions
- **Evaluation is hard** — not everything can be checked with `assert`
- **Instructions are powerful** — you shaped complex behavior without writing complex code